# Preference Signal Detector — V2.1 targeted retraining

This notebook retrains `distilbert-base-multilingual-cased` from the base model
using the combined V2.1 dataset.

Scientific protocol:

1. TRAIN V2.1 = 28,400
2. VALIDATION V2.1 = 5,800
3. select the best checkpoint on VALIDATION macro-F1 at threshold 0.5
4. calibrate a **raw Transformer threshold candidate** on VALIDATION only
5. freeze checkpoint + raw threshold candidate
6. evaluate only already-inspected regression sets:
   - old V2 validation (5,200)
   - old V2 holdout (1,200)
   - legacy V1 test (4,000)
7. export the V2.1 artifact

The fresh final holdout V3 is **not used or loaded anywhere in this notebook**.

Important: the threshold exported here is a raw-Transformer validation candidate.
The final production threshold will be recalibrated locally with
`Transformer + PreferenceContextGuard` before the fresh final holdout is generated.


In [ ]:
!pip install -q "transformers>=4.44,<5" "accelerate>=0.34" "scikit-learn>=1.5" safetensors


In [ ]:
from pathlib import Path
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

SEED = 20260823
BASE_MODEL = "distilbert-base-multilingual-cased"
MAX_LENGTH = 128
BATCH_SIZE = 16
GRAD_ACCUM = 2
MAX_EPOCHS = 5
PATIENCE = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
TARGET_RECALL = 0.985
NUM_WORKERS = 0

WORKING = Path("/kaggle/working")
OUT = WORKING / "preference_signal_detector_v2_1"
BEST_DIR = WORKING / "best_v2_1_checkpoint"
FINAL_ZIP = WORKING / "preference_signal_detector_v2_1.zip"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)


In [ ]:
def find_file(filename: str) -> Path:
    candidates = list(Path("/kaggle/input").rglob(filename))
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename!r} under /kaggle/input, "
            f"found {len(candidates)}: {candidates}"
        )
    return candidates[0]

paths = {
    "train": find_file("preference_signal_train_v2_1.jsonl"),
    "val": find_file("preference_signal_val_v2_1.jsonl"),
    "regression_val_v2": find_file("preference_signal_regression_val_v2_5200.jsonl"),
    "regression_holdout_v2": find_file("preference_signal_regression_holdout_v2_1200.jsonl"),
    "legacy_test": find_file("preference_signal_legacy_test_v1_4000.jsonl"),
}

print(json.dumps({k: str(v) for k, v in paths.items()}, indent=2))

def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(paths["train"])
val_df = load_jsonl(paths["val"])
reg_val_df = load_jsonl(paths["regression_val_v2"])
reg_holdout_df = load_jsonl(paths["regression_holdout_v2"])
legacy_df = load_jsonl(paths["legacy_test"])

for name, df in {
    "train": train_df,
    "val": val_df,
    "reg_val": reg_val_df,
    "reg_holdout": reg_holdout_df,
    "legacy": legacy_df,
}.items():
    assert {"text", "label_id"}.issubset(df.columns), (name, df.columns.tolist())

# Exact protocol checks.
assert len(train_df) == 28400, len(train_df)
assert len(val_df) == 5800, len(val_df)
assert len(reg_val_df) == 5200, len(reg_val_df)
assert len(reg_holdout_df) == 1200, len(reg_holdout_df)
assert len(legacy_df) == 4000, len(legacy_df)

assert train_df.label_id.value_counts().to_dict() in (
    {0: 14200, 1: 14200},
    {1: 14200, 0: 14200},
)
assert val_df.label_id.value_counts().to_dict() in (
    {0: 2900, 1: 2900},
    {1: 2900, 0: 2900},
)

assert train_df["text"].nunique() == 28400
assert val_df["text"].nunique() == 5800

if "group_id" in train_df.columns and "group_id" in val_df.columns:
    assert set(train_df.group_id).isdisjoint(set(val_df.group_id))

assert set(train_df.text).isdisjoint(set(val_df.text))

targeted_train = train_df[train_df.get("is_targeted_v21", False) == True].copy()
targeted_val = val_df[val_df.get("is_targeted_v21", False) == True].copy()

assert len(targeted_train) == 2400, len(targeted_train)
assert len(targeted_val) == 600, len(targeted_val)

print("DATA QA: PASS")
print("train:", len(train_df), train_df.label_id.value_counts().to_dict())
print("val:", len(val_df), val_df.label_id.value_counts().to_dict())
print("targeted train:", len(targeted_train))
print("targeted val:", len(targeted_val))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class PreferenceDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.texts = df["text"].astype(str).tolist()
        self.labels = df["label_id"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = PreferenceDataset(train_df)
val_ds = PreferenceDataset(val_df)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=generator,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "NO_PREFERENCE_SIGNAL", 1: "PREFERENCE_SIGNAL"},
    label2id={"NO_PREFERENCE_SIGNAL": 0, "PREFERENCE_SIGNAL": 1},
).to(device)

print("model ready")


In [ ]:
def binary_metrics(y_true, y_pred, probs=None):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    mp, mr, mf1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "positive_precision": float(p),
        "positive_recall": float(r),
        "positive_f1": float(f1),
        "macro_precision": float(mp),
        "macro_recall": float(mr),
        "macro_f1": float(mf1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) else 0.0,
    }

    if probs is not None and len(set(y_true)) == 2:
        result["roc_auc"] = float(roc_auc_score(y_true, probs))
        result["average_precision"] = float(
            average_precision_score(y_true, probs)
        )

    return result

@torch.no_grad()
def predict_loader(current_model, loader):
    current_model.eval()
    all_labels = []
    all_probs = []

    for batch in loader:
        labels = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = current_model(**batch).logits
        probs = torch.softmax(logits, dim=-1)[:, 1]

        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.detach().cpu().numpy().tolist())

    return np.asarray(all_labels, dtype=int), np.asarray(all_probs, dtype=float)

def metrics_at_threshold(y, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return binary_metrics(y, pred, probs)


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available(),
)

if BEST_DIR.exists():
    shutil.rmtree(BEST_DIR)

best_macro_f1 = -1.0
best_epoch = None
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    started = time.time()

    for step, batch in enumerate(train_loader, start=1):
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available(),
        ):
            outputs = model(**batch, labels=labels)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += float(loss.item()) * GRAD_ACCUM

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    y_val, p_val = predict_loader(model, val_loader)
    val_metrics_05 = metrics_at_threshold(y_val, p_val, 0.5)

    epoch_row = {
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "validation_at_0_5": val_metrics_05,
        "seconds": time.time() - started,
    }
    history.append(epoch_row)

    print(json.dumps(epoch_row, indent=2))

    current = val_metrics_05["macro_f1"]

    if current > best_macro_f1 + 1e-12:
        best_macro_f1 = current
        best_epoch = epoch
        epochs_without_improvement = 0

        model.save_pretrained(BEST_DIR, safe_serialization=True)
        tokenizer.save_pretrained(BEST_DIR / "tokenizer")

        print("NEW BEST:", best_epoch, best_macro_f1)
    else:
        epochs_without_improvement += 1
        print("no improvement:", epochs_without_improvement, "/", PATIENCE)

    if epochs_without_improvement >= PATIENCE:
        print("EARLY STOP")
        break

print("BEST EPOCH:", best_epoch)
print("BEST VALIDATION MACRO-F1 @0.5:", best_macro_f1)


In [ ]:
# Reload the best checkpoint before any threshold calibration.
model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).to(device)

y_val, p_val = predict_loader(model, val_loader)

def choose_exact_threshold(y_true, probs, target_recall=TARGET_RECALL):
    candidates = np.unique(
        np.concatenate([
            np.asarray([0.0, 1.0], dtype=float),
            probs.astype(float),
        ])
    )

    rows = []

    for threshold in candidates:
        metrics = metrics_at_threshold(y_true, probs, float(threshold))
        rows.append({
            "threshold": float(threshold),
            **metrics,
        })

    eligible = [
        row for row in rows
        if row["positive_recall"] >= target_recall
    ]

    if eligible:
        chosen = max(
            eligible,
            key=lambda row: (
                row["positive_precision"],
                row["positive_f1"],
                row["accuracy"],
                row["threshold"],
            ),
        )
        policy = (
            "raw_transformer_max_precision_subject_to_"
            f"recall>={target_recall}"
        )
    else:
        chosen = max(
            rows,
            key=lambda row: (
                row["positive_f1"],
                row["macro_f1"],
                row["accuracy"],
                row["threshold"],
            ),
        )
        policy = "raw_transformer_fallback_max_f1"

    return chosen, policy, len(rows)

chosen, threshold_policy, candidate_count = choose_exact_threshold(
    y_val,
    p_val,
    TARGET_RECALL,
)

RAW_THRESHOLD_CANDIDATE = float(chosen["threshold"])

print("CHECKPOINT + RAW THRESHOLD CANDIDATE NOW FROZEN")
print("best_epoch:", best_epoch)
print("raw threshold candidate:", RAW_THRESHOLD_CANDIDATE)
print("policy:", threshold_policy)
print(json.dumps(chosen, indent=2))


In [ ]:
# Detailed validation metrics and targeted-val metrics after freeze.
val_metrics = metrics_at_threshold(
    y_val,
    p_val,
    RAW_THRESHOLD_CANDIDATE,
)

targeted_mask = val_df.get("is_targeted_v21", False).fillna(False).astype(bool).to_numpy()

targeted_val_metrics = metrics_at_threshold(
    y_val[targeted_mask],
    p_val[targeted_mask],
    RAW_THRESHOLD_CANDIDATE,
)

per_targeted_family = {}

if "hard_family" in val_df.columns:
    targeted_indices = np.where(targeted_mask)[0]
    family_series = val_df.loc[targeted_mask, "hard_family"].astype(str).reset_index(drop=True)

    y_t = y_val[targeted_mask]
    p_t = p_val[targeted_mask]

    for family in sorted(family_series.unique()):
        mask = (family_series.to_numpy() == family)
        per_targeted_family[family] = {
            "n": int(mask.sum()),
            "metrics": metrics_at_threshold(
                y_t[mask],
                p_t[mask],
                RAW_THRESHOLD_CANDIDATE,
            ),
        }

print("VALIDATION V2.1")
print(json.dumps(val_metrics, indent=2))
print("\nTARGETED VALIDATION")
print(json.dumps(targeted_val_metrics, indent=2))
print("\nPER TARGETED FAMILY")
print(json.dumps(per_targeted_family, indent=2))


In [ ]:
class FrameDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].astype(str).tolist()
        self.labels = df["label_id"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def evaluate_frame(df):
    loader = DataLoader(
        FrameDataset(df),
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    y, p = predict_loader(model, loader)
    return metrics_at_threshold(
        y,
        p,
        RAW_THRESHOLD_CANDIDATE,
    )

# These are REGRESSION sets only. They are already inspected.
regression_val_v2_metrics = evaluate_frame(reg_val_df)
regression_holdout_v2_metrics = evaluate_frame(reg_holdout_df)
legacy_test_metrics = evaluate_frame(legacy_df)

print("OLD V2 VALIDATION REGRESSION")
print(json.dumps(regression_val_v2_metrics, indent=2))

print("\nOLD V2 HOLDOUT REGRESSION")
print(json.dumps(regression_holdout_v2_metrics, indent=2))

print("\nLEGACY V1 4000 REGRESSION")
print(json.dumps(legacy_test_metrics, indent=2))


In [ ]:
# Export V2.1 artifact.
if OUT.exists():
    shutil.rmtree(OUT)

(OUT / "model").mkdir(parents=True, exist_ok=True)
(OUT / "tokenizer").mkdir(parents=True, exist_ok=True)

model.save_pretrained(
    OUT / "model",
    safe_serialization=True,
)
tokenizer.save_pretrained(
    OUT / "tokenizer"
)

labels = {
    "id2label": {
        "0": "NO_PREFERENCE_SIGNAL",
        "1": "PREFERENCE_SIGNAL",
    },
    "label2id": {
        "NO_PREFERENCE_SIGNAL": 0,
        "PREFERENCE_SIGNAL": 1,
    },
}

thresholds = {
    "preference_signal_probability_threshold": RAW_THRESHOLD_CANDIDATE,
    "policy": threshold_policy,
    "target_positive_recall": TARGET_RECALL,
    "calibrated_on": "preference_signal_val_v2_1",
    "status": (
        "RAW_TRANSFORMER_VALIDATION_CANDIDATE_ONLY; "
        "final production threshold must be recalibrated locally "
        "with Transformer + PreferenceContextGuard before fresh holdout."
    ),
}

training_config = {
    "version": "v2.1",
    "seed": SEED,
    "base_model": BASE_MODEL,
    "initialization": "fresh_base_model_not_v2_checkpoint",
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "target_recall": TARGET_RECALL,
    "best_epoch": best_epoch,
    "best_validation_macro_f1_at_0_5": best_macro_f1,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "targeted_train_size": len(targeted_train),
    "targeted_val_size": len(targeted_val),
    "regression_val_v2_size": len(reg_val_df),
    "regression_holdout_v2_size": len(reg_holdout_df),
    "legacy_test_size": len(legacy_df),
    "num_workers": NUM_WORKERS,
}

dataset_protocol = {
    "train": "preference_signal_train_v2_1",
    "validation": "preference_signal_val_v2_1",
    "old_v2_validation": "regression only; already inspected",
    "old_v2_holdout": "regression only; already inspected",
    "legacy_test": "preference_signal_legacy_test_v1_4000",
    "known_500": "not used in this notebook; local regression only",
    "fresh_final_holdout_v3": (
        "NOT GENERATED / NOT LOADED / NOT EVALUATED in this notebook. "
        "Generate only after V2.1 checkpoint and FINAL GUARDED threshold are frozen."
    ),
    "threshold_protocol": (
        "Notebook freezes a raw Transformer threshold candidate on V2.1 validation. "
        "Final production threshold is calibrated locally on the final guarded pipeline."
    ),
}

files = {
    "labels.json": labels,
    "thresholds.json": thresholds,
    "training_config.json": training_config,
    "validation_metrics.json": val_metrics,
    "targeted_validation_metrics.json": targeted_val_metrics,
    "targeted_validation_per_family.json": per_targeted_family,
    "regression_val_v2_metrics.json": regression_val_v2_metrics,
    "regression_holdout_v2_metrics.json": regression_holdout_v2_metrics,
    "legacy_test_metrics.json": legacy_test_metrics,
    "training_history.json": history,
    "dataset_protocol.json": dataset_protocol,
}

for filename, payload in files.items():
    (OUT / filename).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in OUT.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                path.relative_to(OUT.parent),
            )

print("FINAL ARTIFACT:", FINAL_ZIP)
print("SIZE MB:", FINAL_ZIP.stat().st_size / (1024**2))
print("\nIMPORTANT:")
print("Do not generate the fresh final holdout yet.")
print("First download this artifact and run local guarded calibration.")


## Download final V2.1 ZIP

This cell uses a relative `FileLink` from `/kaggle/working` to avoid the
absolute-path Jupyter proxy 404 issue.


In [ ]:
import os
from pathlib import Path
from IPython.display import FileLink, display

WORKING_DIR = Path("/kaggle/working")
FINAL_ZIP_NAME = "preference_signal_detector_v2_1.zip"
FINAL_ZIP_PATH = WORKING_DIR / FINAL_ZIP_NAME

if not FINAL_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Final ZIP not found: {FINAL_ZIP_PATH}. "
        "Run the export cell above first."
    )

print(f"Final artifact ready: {FINAL_ZIP_PATH}")
print(f"Size: {FINAL_ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

os.chdir(WORKING_DIR)

display(
    FileLink(
        FINAL_ZIP_NAME,
        result_html_prefix="Download final V2.1 artifact: "
    )
)

print(
    "\nIf the clickable link is blocked by Kaggle, "
    "download /kaggle/working/preference_signal_detector_v2_1.zip "
    "from the Files/Output panel."
)
